# Why are DCs under-produced in the PBISC floor data set? (2026-10-30)
DC-1 extent by group/subtype · DC-2 does the generated pseudobulk follow its own bulk input · DC-3/4 label-free cell-level pDC assembly ·
DC-5 size of the DC signal inside the bulk. Local data only; no new generation.

## DC-1 Extent: group and subtype

In [ ]:
"""DC-1: how big is the DC deficit, in which group, which DC subtype, under which label route."""
import numpy as np, pandas as pd
from scipy import stats
PB="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/pb/"
TR="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/results/big/"
def grp(c,s,o):
    if c=="sepsis":
        if "age_group" in o:   # real
            m={"sepsis":"case","healthy control":"HC","ICU control":"ICU"}; o=o[o.age_group=="adult"].copy(); o["g"]=o.group.map(m)
        else:
            m={"adult_sepsis":"case","adult_HC":"HC","adult_ICU_ctrl":"ICU"}; o=o.copy(); o["g"]=o.group.map(m)
    else:
        if "case_control" in o: o=o.copy(); o["g"]=o.case_control.map({"POSITIVE":"case","NEGATIVE":"HC"})
        else: o=o.copy(); o["g"]=o.group.map({"covid_positive":"case","covid_negative":"HC"})
    return o.dropna(subset=["g"])[["sample","g"]]
rows=[]
for c in ["sepsis","covid"]:
    for s in ["real","floor","prev"]:
        z=np.load(f"{PB}{s}_{c}.npz",allow_pickle=True); o=pd.read_csv(f"{PB}{s}_{c}_obs.csv")
        gr=pd.Series(z["groups"].astype(str)); n=z["n"]
        df=pd.DataFrame({"sample":gr.str.split("\\|\\|").str[0],"six":gr.str.split("\\|\\|").str[1],"n":n})
        w=df.pivot_table(index="sample",columns="six",values="n",aggfunc="sum",fill_value=0)
        w["tot"]=w.sum(1); w["dc_pct"]=100*w.get("DC",0)/w.tot
        w=w.join(grp(c,s,o).set_index("sample"),how="inner")
        for g,d in w.groupby("g"):
            rows.append(dict(cohort=c,source=s,group=g,n_samples=len(d),
                             DC_pct_median=d.dc_pct.median(),DC_pct_mean=d.dc_pct.mean(),
                             pct_samples_DC0=100*(d.get("DC",0)==0).mean(),
                             DC_cells_per_sample_median=float(np.median(d.get("DC",0)))))
A=pd.DataFrame(rows); A.to_csv("dc/dc1_fraction_by_group.csv",index=False)
pd.set_option("display.width",220)
print("=== DC % of all cells per sample (full files; real = author labels, generated = frozen classifier) ===")
print(A.round(3).to_string(index=False))
# ---- subtype via kNN-transferred fine labels on the 60k sets (same route for real B and floor) ----
rows=[]
for c in ["sepsis","covid"]:
    for s in ["realU","floor"]:
        t=pd.read_csv(f"{TR}{c}_{s}_transfer.csv.gz",index_col=0,keep_default_na=False)
        g=t.group.astype(str)
        if c=="sepsis": t["g"]=g.map({"adult_sepsis":"case","adult_HC":"HC","adult_ICU_ctrl":"ICU"})
        else: t["g"]=g.map({"covid_positive":"case","covid_negative":"HC"})
        t=t.dropna(subset=["g"])
        for gg,d in t.groupby("g"):
            per=d.groupby("sample").apply(lambda x: pd.Series({
                "pDC":100*(x.fine_tr=="pDC").mean() if c=="sepsis" else 100*(x.fine_tr=="pDC").mean(),
                "cDC":100*x.fine_tr.isin(["cDC","DC1","DC2","DC3","DC_prolif"]).mean() if c=="sepsis" else 100*x.fine_tr.str.startswith("DC").mean(),
                "ASDC":100*(x.fine_tr=="ASDC").mean(),
                "DC_six_tr":100*(x.six_tr=="DC").mean(),
                "DC_frozen":100*(x.six_type=="DC").mean()}),include_groups=False)
            rows.append(dict(cohort=c,source=s,group=gg,n_samples=len(per),**{k:per[k].median() for k in per.columns},
                             **{k+"_mean":per[k].mean() for k in ["pDC","cDC"]}))
B=pd.DataFrame(rows); B.to_csv("dc/dc1_subtype_transfer.csv",index=False)
print("\n=== DC subtypes, % of all cells per sample (kNN-transferred labels, 60k balanced sets; median) ===")
print(B.round(3).to_string(index=False))


## DC-2 Bulk tracking (same samples)

In [ ]:
"""DC-2: does the generated pseudobulk follow its own bulk input for DC markers?
Same samples: bulk input (x) vs pseudobulk of all generated cells (y). Reference: real sc pseudobulk vs bulk on matched sepsis samples."""
import numpy as np, pandas as pd, pickle
from scipy import stats
PB="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/pb/"
m=pickle.load(open("out2/sym2ens.pkl","rb"))
SETS={"pDC (LILRA4, CLEC4C, PTCRA)":["LILRA4","CLEC4C","PTCRA"],
      "cDC (CD1C, CLEC10A, CLEC9A)":["CD1C","CLEC10A","CLEC9A"],
      "monocyte (CD14, LYZ, VCAN)":["CD14","LYZ","VCAN"],
      "B cell (MS4A1, CD79A, CD19)":["MS4A1","CD79A","CD19"],
      "T cell (CD3E, CD3D, CD2)":["CD3E","CD3D","CD2"],
      "NK (GNLY, KLRF1, NCAM1)":["GNLY","KLRF1","NCAM1"]}
def bulk_lcpm(B):
    L=np.log2(B/B.sum(0)*1e6+1); return L
def bulk_score(L,genes):
    ids=[]; 
    for g in genes:
        cand=[i for i in m.get(g,[]) if i in L.index]
        if cand: ids.append(max(cand,key=lambda i:L.loc[i].mean()))
    return L.loc[ids].mean(0)
def pb_all(name):
    z=np.load(f"{PB}{name}.npz",allow_pickle=True); S=z["S"].astype(float); gr=pd.Series(z["groups"].astype(str))
    samp=gr.str.split("\\|\\|").str[0].values; genes=z["genes"].astype(str)
    df=pd.DataFrame(S,columns=genes); df["s"]=samp; P=df.groupby("s").sum()
    L=np.log2(P.div(P.sum(1),axis=0)*1e6+1); return L
def sc_score(L,genes): return L[[g for g in genes if g in L.columns]].mean(1)
rows=[]; pts=[]
for coh in ["sepsis","covid"]:
    B=pd.read_pickle(f"out2/{coh}_bulk.pkl"); Lb=bulk_lcpm(B)
    for src in ["real","floor","prev"]:
        Lg=pb_all(f"{src}_{coh}")
        common=[s for s in Lg.index if s in Lb.columns]
        if len(common)<10: rows.append(dict(cohort=coh,source=src,n=len(common),note="no sample-level match")); continue
        for k,gs in SETS.items():
            x=bulk_score(Lb[common],gs); y=sc_score(Lg.loc[common],gs)
            r=stats.pearsonr(x,y); sl=np.polyfit(x,y,1)[0]
            rows.append(dict(cohort=coh,source=src,markers=k,n=len(common),r=r.statistic,p=r.pvalue,slope=sl,
                             bulk_mean=x.mean(),sc_mean=y.mean(),bulk_sd=x.std(),sc_sd=y.std()))
            pts.append(pd.DataFrame(dict(cohort=coh,source=src,markers=k,sample=common,bulk=x.values,sc=y.values)))
R=pd.DataFrame(rows); R.to_csv("dc/dc2_bulk_tracking.csv",index=False)
pd.concat(pts).to_csv("dc/dc2_bulk_tracking_points.csv.gz",index=False,compression="gzip")
pd.set_option("display.width",230)
print("=== per-sample: bulk marker score (x) vs pseudobulk of all cells (y), same samples ===")
print(R.dropna(subset=["r"]).pivot_table(index=["cohort","markers"],columns="source",values=["r","slope"]).round(2).to_string())
print("\n(note) COVID real sc samples are date-coded and cannot be matched to bulk samples one-to-one")
# COVID real: patient-level reference
B=pd.read_pickle("out2/covid_bulk.pkl"); cm=pd.read_csv("out2/covid_bulk_meta.csv",index_col=0); Lb=bulk_lcpm(B)
Lg=pb_all("real_covid"); ro=pd.read_csv(PB+"real_covid_obs.csv").set_index("sample")
rows=[]
for k,gs in SETS.items():
    xb=bulk_score(Lb,gs).groupby(cm.loc[Lb.columns,"patient"].values).mean()
    ys=sc_score(Lg,gs); ys=ys.groupby(ro.loc[ys.index,"patient"].astype(str).values).mean()
    c=[p for p in xb.index if p in ys.index]
    r=stats.pearsonr(xb[c],ys[c]); rows.append(dict(markers=k,n_patients=len(c),r=r.statistic,slope=np.polyfit(xb[c],ys[c],1)[0]))
print("\n=== COVID real sc vs bulk, patient-level averages (reference) ===")
print(pd.DataFrame(rows).round(3).to_string(index=False))


## DC-3 Label-free marker co-expression and lineage blending

In [ ]:
"""DC-3: label-free, cell-level. Are pDC/cDC marker UMIs produced? Are they co-expressed in the same cell (identity)?
Where do marker-positive generated cells sit (labels), and do they co-express other lineages' markers?"""
import numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp
IN="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/big/"
TR="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/results/big/"
rng=np.random.default_rng(20261030)
PDC=["LILRA4","CLEC4C","PTCRA","SERPINF1","IL3RA","PLD4"]; PDC_EXCL=["LILRA4","CLEC4C","PTCRA"]
CDC=["CD1C","CLEC10A","CLEC9A","FCER1A"]
BMK=["MS4A1","CD79A","CD79B"]; MMK=["CD14","LYZ","S100A8","S100A9","VCAN"]; TMK=["CD3E","CD3D","IL32"]
rows=[]; lab=[]
for coh in ["sepsis","covid"]:
    R=ad.read_h5ad(f"{IN}{coh}_ref.h5ad"); U=ad.read_h5ad(f"{IN}{coh}_realU.h5ad")
    F=ad.read_h5ad(f"{IN}{coh}_floor.h5ad"); P=ad.read_h5ad(f"{IN}{coh}_prev4k.h5ad")
    gi={g:i for i,g in enumerate(R.var_names)}
    XF=sp.csr_matrix(F.X); sixF=F.obs.six_type.astype(str).to_numpy(); gt=np.asarray(XF.sum(1)).ravel()
    XU=sp.csr_matrix(U.X).astype(np.int64); sixU=U.obs.six_type.astype(str).to_numpy()
    pool={c:gt[sixF==c] for c in np.unique(sixF)}
    tg=np.array([rng.choice(pool.get(c,gt)) for c in sixU],float)
    kp=np.minimum(1.0,tg/np.maximum(np.asarray(XU.sum(1)).ravel(),1))
    XT=XU.copy(); XT.data=rng.binomial(XT.data,np.repeat(kp,np.diff(XT.indptr))); XT.eliminate_zeros()
    trF=pd.read_csv(f"{TR}{coh}_floor_transfer.csv.gz",index_col=0,keep_default_na=False)
    SRC={"real A":(sp.csr_matrix(R.X),R.obs.six_type.astype(str).to_numpy(),R.obs.fine.astype(str).to_numpy()),
         "real B depth-matched":(sp.csr_matrix(XT.astype(np.float32)),sixU,U.obs.fine.astype(str).to_numpy()),
         "PBISC floor":(XF,sixF,trF.fine_tr.astype(str).to_numpy()),
         "PBISC prev K4000":(sp.csr_matrix(P.X),P.obs.six_type.astype(str).to_numpy(),None)}
    def det(X,gs): ii=[gi[g] for g in gs if g in gi]; return (X[:,ii]>0).toarray()
    for s,(X,six,fine) in SRC.items():
        n=X.shape[0]
        Dp=det(X,PDC_EXCL); Dc=det(X,CDC)
        any_p=Dp.any(1); co_p=Dp.sum(1)>=2
        any_c=Dc.any(1); co_c=Dc.sum(1)>=2
        b=det(X,BMK).any(1); mo=det(X,MMK).any(1); t=det(X,TMK).any(1)
        rows.append(dict(cohort=coh,source=s,n_cells=n,
            pct_any_pDCmarker=100*any_p.mean(),pct_pDC_coexpr2=100*co_p.mean(),
            coexpr_given_any_pDC=100*co_p.sum()/max(any_p.sum(),1),
            pct_any_cDCmarker=100*any_c.mean(),pct_cDC_coexpr2=100*co_c.mean(),
            coexpr_given_any_cDC=100*co_c.sum()/max(any_c.sum(),1),
            pDCco_with_B=100*(co_p&b).sum()/max(co_p.sum(),1),
            pDCco_with_Mono=100*(co_p&mo).sum()/max(co_p.sum(),1),
            pDCco_with_T=100*(co_p&t).sum()/max(co_p.sum(),1),
            cDCco_with_B=100*(co_c&b).sum()/max(co_c.sum(),1),
            cDCco_with_Mono=100*(co_c&mo).sum()/max(co_c.sum(),1),
            cDCco_with_T=100*(co_c&t).sum()/max(co_c.sum(),1)))
        # where do single-marker pDC+ cells sit (lineage label)?
        vc=pd.Series(six[any_p]).value_counts(normalize=True)*100
        for k,v in vc.items(): lab.append(dict(cohort=coh,source=s,population="any pDC-exclusive marker",label=k,pct=v))
        vc=pd.Series(six[co_p]).value_counts(normalize=True)*100
        for k,v in vc.items(): lab.append(dict(cohort=coh,source=s,population="≥2 pDC-exclusive markers",label=k,pct=v))
T=pd.DataFrame(rows); T.to_csv("dc/dc3_cell_level.csv",index=False)
L=pd.DataFrame(lab); L.to_csv("dc/dc3_marker_cell_labels.csv",index=False)
pd.set_option("display.width",260)
print("=== % of ALL cells (label-free) ===")
print(T[["cohort","source","pct_any_pDCmarker","pct_pDC_coexpr2","coexpr_given_any_pDC","pct_any_cDCmarker","pct_cDC_coexpr2","coexpr_given_any_cDC"]].round(3).to_string(index=False))
print("\n=== lineage blending: among co-expressing DC-like cells, % that also carry another lineage's markers ===")
print(T[["cohort","source","pDCco_with_B","pDCco_with_Mono","pDCco_with_T","cDCco_with_B","cDCco_with_Mono","cDCco_with_T"]].round(1).to_string(index=False))
print("\n=== label of cells carrying pDC-exclusive marker UMIs ===")
print(L.pivot_table(index=["cohort","population","label"],columns="source",values="pct").round(1).fillna(0).to_string())


## DC-4 LILRA4 assembly and composition tracking

In [ ]:
"""DC-4: (a) LILRA4+ cells: are they assembled pDCs (co-express the pDC programme)? (b) per-sample DC % vs bulk DC signal."""
import numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp, pickle
from scipy import stats
IN="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/big/"
PB="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/pb/"
rng=np.random.default_rng(20261030)
PROG=["CLEC4C","IRF7","TCF4","SPIB","IRF8","GZMB","PLD4","SERPINF1","IL3RA","ITM2C","PTCRA"]
rows=[]
for coh in ["sepsis","covid"]:
    R=ad.read_h5ad(f"{IN}{coh}_ref.h5ad"); U=ad.read_h5ad(f"{IN}{coh}_realU.h5ad")
    F=ad.read_h5ad(f"{IN}{coh}_floor.h5ad"); P=ad.read_h5ad(f"{IN}{coh}_prev4k.h5ad")
    gi={g:i for i,g in enumerate(R.var_names)}
    XF=sp.csr_matrix(F.X); sixF=F.obs.six_type.astype(str).to_numpy(); gt=np.asarray(XF.sum(1)).ravel()
    XU=sp.csr_matrix(U.X).astype(np.int64); sixU=U.obs.six_type.astype(str).to_numpy()
    pool={c:gt[sixF==c] for c in np.unique(sixF)}; tg=np.array([rng.choice(pool.get(c,gt)) for c in sixU],float)
    kp=np.minimum(1.0,tg/np.maximum(np.asarray(XU.sum(1)).ravel(),1)); XT=XU.copy()
    XT.data=rng.binomial(XT.data,np.repeat(kp,np.diff(XT.indptr))); XT.eliminate_zeros()
    SRC={"real A":(sp.csr_matrix(R.X),R.obs.six_type.astype(str).to_numpy()),
         "real B depth-matched":(sp.csr_matrix(XT.astype(np.float32)),sixU),
         "PBISC floor":(XF,sixF),"PBISC prev K4000":(sp.csr_matrix(P.X),P.obs.six_type.astype(str).to_numpy())}
    for s,(X,six) in SRC.items():
        L=(X[:,gi["LILRA4"]]>0).toarray().ravel()
        prog=(X[:,[gi[g] for g in PROG if g in gi]]>0).toarray()
        nprog=prog.sum(1)
        umi=X[:,gi["LILRA4"]].toarray().ravel()
        rows.append(dict(cohort=coh,source=s,
            pct_LILRA4pos=100*L.mean(),
            LILRA4_UMI_in_pos=float(umi[L].mean()) if L.any() else np.nan,
            pDCprog_genes_in_LILRA4pos=float(nprog[L].mean()) if L.any() else np.nan,
            pDCprog_genes_in_LILRA4neg=float(nprog[~L].mean()),
            pct_LILRA4pos_labelled_DC=100*(six[L]=="DC").mean() if L.any() else np.nan,
            pct_assembled_pDC=100*((L)&(nprog>=4)).mean()))
A=pd.DataFrame(rows); A.to_csv("dc/dc4_lilra4.csv",index=False)
pd.set_option("display.width",240)
print("=== LILRA4 (most pDC-exclusive gene): produced? assembled into a pDC? ===")
print(A.round(3).to_string(index=False))

# ---- (b) composition tracking: per-sample DC % vs bulk DC marker score ----
m=pickle.load(open("out2/sym2ens.pkl","rb"))
def bscore(L,gs):
    ids=[max([i for i in m.get(g,[]) if i in L.index],key=lambda i:L.loc[i].mean()) for g in gs if any(i in L.index for i in m.get(g,[]))]
    return L.loc[ids].mean(0)
out=[]
for coh in ["sepsis"]:
    B=pd.read_pickle(f"out2/{coh}_bulk.pkl"); Lb=np.log2(B/B.sum(0)*1e6+1)
    xs=bscore(Lb,["LILRA4","CLEC4C","PTCRA","CD1C","CLEC10A","CLEC9A"])
    for src in ["real","floor","prev"]:
        z=np.load(f"{PB}{src}_{coh}.npz",allow_pickle=True); gr=pd.Series(z["groups"].astype(str))
        df=pd.DataFrame({"s":gr.str.split("\\|\\|").str[0],"six":gr.str.split("\\|\\|").str[1],"n":z["n"]})
        w=df.pivot_table(index="s",columns="six",values="n",aggfunc="sum",fill_value=0)
        dc=100*w["DC"]/w.sum(1); c=[s for s in dc.index if s in xs.index]
        r=stats.spearmanr(xs[c],dc[c])
        out.append(dict(cohort=coh,source=src,n=len(c),spearman_bulkDCscore_vs_DCpct=r.statistic,p=r.pvalue,
                        DCpct_range_p10_p90=f"{np.percentile(dc[c],10):.2f}–{np.percentile(dc[c],90):.2f}"))
O=pd.DataFrame(out); O.to_csv("dc/dc4_composition_tracking.csv",index=False)
print("\n=== per-sample DC % vs bulk DC marker score (sepsis, same samples) ===")
print(O.round(3).to_string(index=False))


## DC-5 DC signal size inside the bulk

In [ ]:
"""DC-5: how large is the DC signal inside the bulk input, compared with other lineages' markers?"""
import numpy as np, pandas as pd, pickle
m=pickle.load(open("out2/sym2ens.pkl","rb"))
G={"pDC":["LILRA4","CLEC4C","PTCRA"],"cDC":["CD1C","CLEC10A","CLEC9A"],
   "monocyte":["CD14","LYZ","VCAN"],"B cell":["MS4A1","CD79A","CD19"],"T cell":["CD3E","CD3D","CD2"],"NK":["GNLY","KLRF1","NCAM1"]}
rows=[]
for coh in ["sepsis","covid"]:
    B=pd.read_pickle(f"out2/{coh}_bulk.pkl"); C=B/B.sum(0)*1e6
    for k,gs in G.items():
        ids=[max([i for i in m.get(g,[]) if i in C.index],key=lambda i:C.loc[i].mean()) for g in gs if any(i in C.index for i in m.get(g,[]))]
        v=C.loc[ids].sum(0)
        rows.append(dict(cohort=coh,lineage=k,marker_CPM_median=v.median(),
                         share_of_bulk_pct=100*v.median()/1e6,
                         cv_across_samples=v.std()/v.mean()))
S=pd.DataFrame(rows); S.to_csv("dc/dc5_bulk_signal_size.csv",index=False)
pd.set_option("display.width",200)
p=S.pivot_table(index="lineage",columns="cohort",values=["marker_CPM_median","cv_across_samples"]).round(2)
print(p.loc[["T cell","monocyte","NK","B cell","cDC","pDC"]].to_string())
